# Phonopy calculation

#Calculation Setup

The phonon calculation uses the janus-core/ phonopy interface to calculate the force constants and return the information contained within the phonopy yaml file. Prior to running a phonon calculation using aiida-mlip you need to define some inputs as AiiDA data types, to then pass them to the calculation. The first few steps will follow those discussed in the single point calculation.
First of all we need a structure on which to perform the calculations. Again we will use the NaCl structure that we define using ASE, or alternatively one can choose one of the structures in the folder `Structures`.
The input structure in aiida-mlip needs to be saved as a StructureData type:

In [1]:
from aiida import load_profile
load_profile()

Profile<uuid='a0bc4bdaef58416f840387aceea091a9' name='john'>

In [2]:
from aiida.orm import StructureData
from ase.build import bulk
from ase.io import read

#structure = StructureData(ase=read("Structures/qmof-ffeef76.cif"))
structure = StructureData(ase=bulk("NaCl", "rocksalt", 5.63))

Then we need to choose a model and architecture to be used for the calculation and save it as ModelData type, a specific data type of this plugin.
In this example we use MACE with a model that we download from this URL: "https://github.com/stfc/janus-core/raw/main/tests/models/mace_mp_small.model", and we save the file in the cache folder (default="~/.cache/mlips/"):


In [3]:
from aiida_mlip.data.model import ModelData
#uri = "https://github.com/stfc/janus-core/raw/main/tests/models/mace_mp_small.model"
#model = ModelData.from_uri(uri, architecture="mace_mp", cache_dir="mlips")

If we already have the model saved in some folder we can save it as:

In [4]:
model = ModelData.from_local("/mnt/87bd3caf-192d-48ed-9f63-c5afcf32b9ec/ML/janus_work/models/MACE-matpes-r2scan-omat-ft.model", architecture="mace")

Another parameter that we need to define as AiiDA type is the code. Assuming the code is saved as `janus` in the `localhost` computer, the code info that are needed can be loaded as follow:


In [5]:
from aiida.orm import load_code
code = load_code("janus@localhost")

The other inputs can be set up as AiiDA Str. There is a default for every input except the structure and code. This is a list of possible inputs:

In [14]:
from aiida.orm import Dict, Str, Bool

inputs = {
        "code": code,
        "model": model,
        "struct": structure,
        "arch": Str(model.architecture),
        "device": Str("cpu"),
        "supercell": Str("2 2 2"),
        "minimize": False,
        "calc_kwargs": Dict({"dispersion": False}),
        "metadata": {"options": {"resources": {"num_machines": 1}}},
    }

# Phonon Calculation Setup

The calculation must be set as in previous notebooks, except that we use `mlip.ph`:

In [7]:
from aiida.plugins import CalculationFactory
phononCalc = CalculationFactory("mlip.ph")

It is now possible to run the phonon calculation using the Aiida platform:


In [15]:
from aiida.engine import run_get_node
result, node = run_get_node(phononCalc, inputs)

`result` is a dictionary of the available results obtained from the calculation, while node contains the infor on the node where the calculation is run:


In [16]:
print(result)
print(node)

{'remote_folder': <RemoteData: uuid: bc7b88b8-a6fc-4f0d-8cd9-340c56fd12b3 (pk: 2909)>, 'retrieved': <FolderData: uuid: 08d814a5-a9d9-49a3-98a1-0112caed5037 (pk: 2910)>, 'log_output': <SinglefileData: uuid: 71d26407-3eba-4b69-b1de-99d2c4a09067 (pk: 2911)>, 'std_output': <SinglefileData: uuid: 6aef4b7c-295b-430e-9e04-0e55ebec0819 (pk: 2912)>, 'xyz_output': <SinglefileData: uuid: ac2ff864-0414-4a35-9192-0d53d0c69582 (pk: 2913)>, 'results_dict': <Dict: uuid: 38ac778e-2e84-41ea-9d50-c23ec52e214d (pk: 2914)>}
uuid: a7ffd153-f8f4-425c-886f-4d0dd76b3c8e (pk: 2908) (aiida.calculations:mlip.ph)


We can check if the calculation finished with errors. If everything worked the exit code should be 0

In [17]:
if node.is_finished_ok:
     print(f"Caculation is finished without errors with exit status {node.exit_status}")
else:
     print(f"Some errors occurred with exit status {node.exit_status}") 

Caculation is finished without errors with exit status 0


If the job fails, it is possible to open a new shell in the aiida work directory using `verdi calcjob gotocomputer node.pk`. The `node.pk` is the PK value of the node used in the calculation. The files in this directory, especially those assocociated with errors/logfs, often have valuable information as to why the job has failed.

If more information are needed on specific outputs they can be called like:

In [18]:
print(result["results_dict"].get_dict())

{'phonopy': {'version': '2.47.1', 'frequency_unit_conversion_factor': 15.633302, 'symmetry_tolerance': 1e-05}, 'space_group': {'type': 'Fm-3m', 'number': 225, 'Hall_symbol': '-F 4 2 3'}, 'supercell_matrix': [[2, 0, 0], [0, 2, 0], [0, 0, 2]], 'primitive_cell': {'lattice': [[0.0, 2.815, 2.815], [2.815, 0.0, 2.815], [2.815, 2.815, 0.0]], 'points': [{'symbol': 'Na', 'coordinates': [0.0, 0.0, 0.0], 'mass': 22.989769}, {'symbol': 'Cl', 'coordinates': [0.5, 0.5, 0.5], 'mass': 35.45}], 'reciprocal_lattice': [[-0.17761989342806, 0.17761989342806, 0.17761989342806], [0.17761989342806, -0.17761989342806, 0.17761989342806], [0.17761989342806, 0.17761989342806, -0.17761989342806]]}, 'unit_cell': {'lattice': [[0.0, 2.815, 2.815], [2.815, 0.0, 2.815], [2.815, 2.815, 0.0]], 'points': [{'symbol': 'Na', 'coordinates': [0.0, 0.0, 0.0], 'mass': 22.989769, 'reduced_to': 1}, {'symbol': 'Cl', 'coordinates': [0.5, 0.5, 0.5], 'mass': 35.45, 'reduced_to': 2}]}, 'supercell': {'lattice': [[0.0, 5.63, 5.63], [5.63

The most useful information, e.g. force constants, supercell matrix, atom displacements, is the `phonopy.yaml` dictionary. This can be obtained using 

In [19]:
print(result["results_dict"].get_dict()) #warning this is a large output

{'phonopy': {'version': '2.47.1', 'frequency_unit_conversion_factor': 15.633302, 'symmetry_tolerance': 1e-05}, 'space_group': {'type': 'Fm-3m', 'number': 225, 'Hall_symbol': '-F 4 2 3'}, 'supercell_matrix': [[2, 0, 0], [0, 2, 0], [0, 0, 2]], 'primitive_cell': {'lattice': [[0.0, 2.815, 2.815], [2.815, 0.0, 2.815], [2.815, 2.815, 0.0]], 'points': [{'symbol': 'Na', 'coordinates': [0.0, 0.0, 0.0], 'mass': 22.989769}, {'symbol': 'Cl', 'coordinates': [0.5, 0.5, 0.5], 'mass': 35.45}], 'reciprocal_lattice': [[-0.17761989342806, 0.17761989342806, 0.17761989342806], [0.17761989342806, -0.17761989342806, 0.17761989342806], [0.17761989342806, 0.17761989342806, -0.17761989342806]]}, 'unit_cell': {'lattice': [[0.0, 2.815, 2.815], [2.815, 0.0, 2.815], [2.815, 2.815, 0.0]], 'points': [{'symbol': 'Na', 'coordinates': [0.0, 0.0, 0.0], 'mass': 22.989769, 'reduced_to': 1}, {'symbol': 'Cl', 'coordinates': [0.5, 0.5, 0.5], 'mass': 35.45, 'reduced_to': 2}]}, 'supercell': {'lattice': [[0.0, 5.63, 5.63], [5.63

Or saved to a yaml or json file for later use in Phonopy or Euphonic for instance

In [20]:
import yaml
with open('data.yml', 'w') as file:
    yaml.dump(result["results_dict"].get_dict(), file)

import json
file_name = "data.json"
with open(file_name, "w") as file:
    json.dump(result["results_dict"].get_dict(), file, indent=4)  # (make it more human readable with the indent


Through the command line we can see the processes that are run

In [ ]:
! verdi process list -a

And see the results we are interested in. Substitute the number with the PK number of your calculation

In [ ]:
! verdi calcjob res pk

We can also see the inputs and outputs of the calculation

In [ ]:
! verdi node show pk  